In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D 
from matplotlib import gridspec

import scanpy as sc

import os

import scvi

import seaborn as sns



/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing CSCDataset from `anndata.experimental` is deprecated. Import anndata.abc.CSCDataset instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndat

# Load Data

## Load entropy results

In [2]:
# Load Entropy value for each barcode on atac (if any barcode has invalid entropy value, it's not included in the file)
entropy_dir = '/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F'
entropy_df_file = os.path.join(entropy_dir, 'calculated_barcode_entropy_df.tsv')
entropy_df = pd.read_csv(entropy_df_file, sep=',', index_col=0)
entropy_df['log10_Entropy'] = np.log10(entropy_df['Entropy'])


In [3]:
# Load Cell Ranger  RNA filtering criteria barcodes
CR_rnamodality_BC_file = '/mnt/hdd_bob/syy/adipose/atac/protocol_benchmark/cr_results/rna/VIB_10xmultiome_2_rna/outs/filtered_feature_bc_matrix/barcodes.tsv'
CR_rnamodality_bc = pd.read_csv(CR_rnamodality_BC_file, header=None, sep='\t', names=['barcodes'])

CR_rnamodality_bc['rna_bc'] = CR_rnamodality_bc['barcodes'].map(lambda x: x.split('-1')[0])


In [8]:
# Load Cell Ranger called cell barcodes (using ATAC modality alone within Cell Ranger pipeline)
CR_atacmodality_BC_file ='/mnt/hdd_bob/syy/adipose/atac/protocol_benchmark/cr_results/atac/VIB_10xmultiome_2/outs/filtered_peak_bc_matrix/barcodes.tsv'
CR_atacmodality_bc = pd.read_csv(CR_atacmodality_BC_file, header=None, sep='\t', names=['barcodes'])


# Load ArchR-TSS called cell barcodes 
archr_cell_calling_file = os.path.join(entropy_dir, '_ArchR_TSS/TSS_passed_bc.tsv')
archr_Tss_cell_call = pd.read_csv(archr_cell_calling_file, sep='\t', header=0)
archr_Tss_cell_call = archr_Tss_cell_call.loc[archr_Tss_cell_call['Keep'] == 1 ]

# Load SADE/Entropy called cell barcodes
entropy_filtered_bc_file = os.path.join(entropy_dir, 'entropy_filtered_bc_df.tsv')
entropy_filtered_bc_df = pd.read_csv(entropy_filtered_bc_file, sep=',', index_col=0)


print ('----------RNA modality ----------')
print('Total number of barcodes identified as cells by Cell Ranger RNA: ', CR_rnamodality_bc.shape[0])
print ('----------ATAC modality based cell calling methods----------')
print('Total number of barcodes identified as cells by ArchR/TSS: ', archr_Tss_cell_call.shape[0])
print('Total number of barcodes identified as cells by Cell Ranger/FRIP ATAC: ', CR_atacmodality_bc.shape[0])
print('Total number of barcodes identified as cells by SADE/Entropy filtering: ', entropy_filtered_bc_df.shape[0])

----------RNA modality ----------
Total number of barcodes identified as cells by Cell Ranger RNA:  2261
----------ATAC modality based cell calling methods----------
Total number of barcodes identified as cells by ArchR/TSS:  2132
Total number of barcodes identified as cells by Cell Ranger/FRIP ATAC:  1959
Total number of barcodes identified as cells by SADE/Entropy filtering:  2101


In [9]:
entropy_df['atac_bc'] = entropy_df.index.map(lambda x: x.split('-1')[0])

archr_Tss_cell_call['atac_bc'] = archr_Tss_cell_call['cellNames'].apply(lambda x: x.split('#')[1].split('-')[0])
CR_atacmodality_bc['atac_bc'] = CR_atacmodality_bc['barcodes'].map(lambda x: x.split('-1')[0])
entropy_filtered_bc_df['atac_bc'] = entropy_filtered_bc_df.index.map(lambda x: x.split('-1')[0])

In [10]:
# Load  atac-rna-barcode map 
bc_map_file = '/home/syyang/adipose_ln/multiom/atac_rna_barcodes_map.tsv'
bc_map_pd = pd.read_csv(bc_map_file, sep='\t')

## Given some barcodes could have invalid entropy and does not have a record -- load fragment file 

In [11]:
frag_file = os.path.join(entropy_dir, 'fragments.tsv')
frag = pd.read_csv(frag_file, sep='\t', index_col=3, header=None)
frag.index.name ='atac_BC'
frag.columns = ['chrom', 'start', 'end', 'support']
frag['atac_bc'] = frag.index.map(lambda x: x.split('-1')[0])
frag['frag_length'] = frag['end'] - frag['start']



In [12]:
_total_frag_per_bc = frag.groupby('atac_bc').size().to_frame()
_total_frag_per_bc.columns = ['total_fragments']
_total_frag_per_bc.head()

,total_fragments
atac_bc,
AAACAAGCAAACATGT,4
AAACAAGCAAACCAGC,1
AAACAAGCAAACCTAG,207
AAACAAGCAAACTAAC,1
AAACAAGCAAAGAAGC,987


In [13]:
_total_frag_per_bc['atac_pass_CR'] = _total_frag_per_bc.index.isin(CR_atacmodality_bc['atac_bc'])
_total_frag_per_bc['atac_pass_entropy'] = _total_frag_per_bc.index.isin(entropy_filtered_bc_df['atac_bc'])
_total_frag_per_bc['atac_pass_archr_TSS'] = _total_frag_per_bc.index.isin(archr_Tss_cell_call['atac_bc'])


In [14]:
_total_frag_per_bc['_2set_identified_by_atac_SC'] = _total_frag_per_bc.apply(
    lambda row: 'both' if row['atac_pass_entropy'] and row['atac_pass_CR'] else 
                ('entropy_only' if row['atac_pass_entropy'] and not row['atac_pass_CR'] else 
                 ('cr_only' if not row['atac_pass_entropy'] and row['atac_pass_CR'] else 'none')), axis=1
)


In [15]:
_total_frag_per_bc['_2set_identified_by_atac_ST'] = _total_frag_per_bc.apply(
    lambda row: 'both' if row['atac_pass_entropy'] and row['atac_pass_archr_TSS'] else 
                ('entropy_only' if row['atac_pass_entropy'] and not row['atac_pass_archr_TSS'] else 
                 ('tss_only' if not row['atac_pass_entropy'] and row['atac_pass_archr_TSS'] else 'none')), axis=1
)

In [16]:
if 'rna_barcodes' not in _total_frag_per_bc.columns:
    _total_frag_per_bc = _total_frag_per_bc.merge(bc_map_pd.set_index('atac_barcodes'), left_index=True, right_index=True, how='left')
    _total_frag_per_bc['rna_pass_CR'] = _total_frag_per_bc['rna_barcodes'].isin(CR_rnamodality_bc['rna_bc']) 
    
_total_frag_per_bc['rna_pass_CR'].value_counts()

rna_pass_CR
False    286748
True       2261
Name: count, dtype: int64

In [17]:
total_frag_per_bc = _total_frag_per_bc.loc[_total_frag_per_bc['atac_pass_CR'] | _total_frag_per_bc['atac_pass_entropy'] | _total_frag_per_bc['atac_pass_archr_TSS']].copy()
print('Total number of barcodes that pass at least one of the three ATAC-based filtering methods: ', total_frag_per_bc.shape[0])

Total number of barcodes that pass at least one of the three ATAC-based filtering methods:  2384


In [18]:
total_frag_per_bc['rna_pass_CR'].value_counts()

rna_pass_CR
True     2044
False     340
Name: count, dtype: int64

In [19]:
total_frag_per_bc.groupby([ 'atac_pass_entropy', 'rna_pass_CR']).size().fillna(0).unstack()

rna_pass_CR,False,True
atac_pass_entropy,,
False,211,72
True,129,1972


In [20]:
total_frag_per_bc.groupby([ 'atac_pass_CR', 'rna_pass_CR']).size().fillna(0).unstack()

rna_pass_CR,False,True
atac_pass_CR,,
False,272,153
True,68,1891


In [21]:
total_frag_per_bc.groupby([ 'atac_pass_archr_TSS', 'rna_pass_CR']).size().fillna(0).unstack()

rna_pass_CR,False,True
atac_pass_archr_TSS,,
False,104,148
True,236,1896


In [22]:
total_frag_per_bc['Entropy'] = total_frag_per_bc.index.map(entropy_df.set_index('atac_bc')['Entropy'])

In [23]:
# Save dataframe 
Union_cell_subdir = os.path.join(entropy_dir, '_cell_calling_comparison')
Union_cell_info_file = os.path.join(Union_cell_subdir, 'Union_cell_3set.tsv')
total_frag_per_bc.to_csv(Union_cell_info_file, sep='\t')